In [ ]:
from samap import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder,
                            sankey_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samap.sam import SAM
import pandas as pd

import scanpy as sc
import anndata as ad
import numpy as np
from pathlib import Path
import scipy.sparse as sp
from samap.utils import save_samap, load_samap
import matplotlib.colors as mcolors

In [ ]:
import numpy as np, pandas as pd, scanpy as sc
from pathlib import Path

PROJECT_DIR = Path("~/BH2026-23-single-cell/WP2")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed_h5ad"
MOUSE_ANNOTATED_PATH = PROCESSED_DIR / "mouse_annot_SKM_samap_ready_sub.h5ad"
ZEBRAFISH_READY_PATH = PROCESSED_DIR / "zebrafish_SKM_samap_ready_sub.h5ad"
ORTHO_TSV = Path("~/BH2026-23-single-cell/WP2/data/mm__v__dr.tsv")            
MAPS_OF_DIR = PROJECT_DIR / "maps_orthofinder"

ortho = pd.read_csv(ORTHO_TSV, sep="\t")
print(ortho.columns.tolist())                         

ortho["mm_id"] = ortho["mm"].str.split(", ")
ortho["dr_id"] = ortho["dr"].str.split(", ")
n_mm, n_dr = ortho.mm_id.str.len(), ortho.dr_id.str.len()
ortho["type"] = np.select([(n_mm == 1) & (n_dr == 1), n_mm == 1, n_dr == 1],
                          ["1:1", "1:many", "many:1"], "many:many")

pairs = ortho.explode("mm_id").explode("dr_id")[["mm_id", "dr_id", "type"]]
for c in ["mm_id", "dr_id"]:
    pairs[c] = pairs[c].str.replace(r"\.\d+$", "", regex=True)  

print(pairs.type.value_counts())

In [ ]:
mm_var = sc.read_h5ad(MOUSE_ANNOTATED_PATH, backed="r").var
dr_var = sc.read_h5ad(ZEBRAFISH_READY_PATH, backed="r").var

def id2gene(var):
    ids = var["ensembl_gene_id"].astype(str).str.replace(r"\.\d+$", "", regex=True)
    return dict(zip(ids, var.index))        # index = samap_gene

mm_map, dr_map = id2gene(mm_var), id2gene(dr_var)

print("mouse IDs found:", pairs.mm_id.isin(mm_map).mean())
print("fish IDs found: ", pairs.dr_id.isin(dr_map).mean())

pairs["mm_gene"] = pairs.mm_id.map(mm_map)
pairs["dr_gene"] = pairs.dr_id.map(dr_map)
pairs = pairs.dropna(subset=["mm_gene", "dr_gene"]).drop_duplicates(["mm_gene", "dr_gene"])
print(len(pairs), "pairs;", pairs.mm_gene.nunique(), "mouse genes;", pairs.dr_gene.nunique(), "fish genes")

In [ ]:
out = MAPS_OF_DIR / "mmdr"         
out.mkdir(parents=True, exist_ok=True)

def write_map(q, s, path):
    n = len(q)
    df = pd.DataFrame({0: q.values, 1: s.values,
                       2: [100.0]*n, 3: [100]*n, 4: [0]*n, 5: [0]*n,
                       6: [1]*n, 7: [100]*n, 8: [1]*n, 9: [100]*n,
                       10: [1e-50]*n, 11: [100.0]*n})    # e-value, bitscore
    df.to_csv(path, sep="\t", header=False, index=False)

write_map(pairs.mm_gene, pairs.dr_gene, out / "mm_to_dr.txt")
write_map(pairs.dr_gene, pairs.mm_gene, out / "dr_to_mm.txt")

In [ ]:
MAPS_DIR = "~/BH2026-23-single-cell/WP2/maps_orthofinder"

fn1 = str(MOUSE_ANNOTATED_PATH)
fn2 = str(ZEBRAFISH_READY_PATH)         
filenames = {'mm': fn1, 'dr': fn2}        
keys = {'mm': 'cell_type', 'dr': 'cell_type'}

In [ ]:
sm = SAMAP(filenames, f_maps=str(MAPS_DIR) + "/") #loads sam objects directly

In [ ]:
sm.run()
samap = sm.samap   # SAM object with 2 species

In [ ]:
D, MappingTable = get_mapping_scores(sm, keys, n_top=0)

In [ ]:
SAMAP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df = pd.DataFrame(D)
df1 = pd.DataFrame(MappingTable)
df.to_csv(SAMAP_OUTPUT_DIR / "MappingD_mm_dr_annot_orthofinder.csv", index=False)
df1.to_csv(SAMAP_OUTPUT_DIR / "MappingT_mm_dr_annot_orthofinder.csv")

In [ ]:
D.head()
MappingTable.iloc[:3, :3]

In [ ]:
# Combined AnnData and SAMap object is the full sm

SAMAP_H5AD_PATH = SAMAP_OUTPUT_DIR / "samap_mm_dr_annot_orthofinder.h5ad"
SAMAP_OBJECT_PATH = SAMAP_OUTPUT_DIR / "SamapObject_mm_dr_annot_orthofinder"

sm.samap.adata.write(SAMAP_H5AD_PATH)
save_samap(sm, str(SAMAP_OBJECT_PATH))

print(SAMAP_H5AD_PATH.exists(), SAMAP_OBJECT_PATH.exists())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

FIG_DIR = SAMAP_OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

adata = sm.samap.adata
print(adata.obs["species"].value_counts())

In [ ]:
is_mm = adata.obs["species"].astype(str) == "mm"
adata.obs["joint_cell_type"] = np.where(
    is_mm,
    "mm: " + adata.obs["mm_cell_type"].astype(str),
    "dr: " + adata.obs["dr_cell_type"].astype(str),
)
adata.obs["joint_cell_type"] = adata.obs["joint_cell_type"].astype("category")

sc.pl.umap(adata, color="joint_cell_type", frameon=False, size=8, show=False)
plt.savefig(FIG_DIR / "umap_joint_celltype_orthofinder.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
mm_vs_dr = MappingTable.loc[
    [i for i in MappingTable.index if i.startswith("mm_")],
    [c for c in MappingTable.columns if c.startswith("dr_")]
]

hm = mm_vs_dr.copy()
hm.index = hm.index.str.replace(r"^mm_", "", regex=True)
hm.columns = hm.columns.str.replace(r"^dr_", "", regex=True)
hm = hm.loc[hm.max(axis=1) > 0, hm.max(axis=0) > 0]

g = sns.clustermap(
    hm,
    cmap="viridis", vmin=0, vmax=hm.values.max(),
    figsize=(0.45 * hm.shape[1] + 4, 0.35 * hm.shape[0] + 3),
    linewidths=0.3,
    annot=hm.size <= 400, fmt=".2f", annot_kws={"size": 7},
    cbar_kws={"label": "SAMap mapping score"},
)
g.ax_heatmap.set_xlabel("Zebrafish cell types")
g.ax_heatmap.set_ylabel("Mouse cell types")
g.savefig(FIG_DIR / "heatmap_mm_vs_dr_orthofinder.png", dpi=300, bbox_inches="tight")
g.savefig(FIG_DIR / "heatmap_mm_vs_dr_orthofinder.pdf", bbox_inches="tight")
plt.show()